In [1]:
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names
import random

def get_random_mmlu_samples(num_samples=100):
    """
    从 MMLU (cais/mmlu) 数据集的所有科目中随机抽取指定数量的样本。
    """
    try:
        # 1. 获取 MMLU 数据集的所有配置（即所有科目）
        print("正在获取 MMLU 科目列表...")
        all_subjects = get_dataset_config_names("cais/mmlu")
        
        # 'all' 是一个特殊的配置名，我们通常不单独加载它
        if "all" in all_subjects:
            all_subjects.remove("all")
        
        print(f"共找到 {len(all_subjects)} 个科目。")

        # 2. 加载每个科目的 'test' split，并添加科目名称
        all_splits = []
        print("正在加载所有科目的 'test' 数据集...")
        for subject in all_subjects:
            try:
                # 加载 test split
                ds = load_dataset("cais/mmlu", subject, split="test")
                
                # 添加一个 'subject' 列，以便知道问题来自哪个科目
                # 使用 map 函数来添加新列
                def add_subject_column(example):
                    example['subject'] = subject
                    return example
                
                ds = ds.map(add_subject_column)
                all_splits.append(ds)
                
            except Exception as e:
                print(f"加载科目 {subject} 失败: {e}")

        if not all_splits:
            print("没有成功加载任何数据集。")
            return

        # 3. 将所有数据集拼接成一个大的数据集
        print("正在合并所有数据集...")
        combined_dataset = concatenate_datasets(all_splits)
        print(f"数据集合并完毕，总问题数: {len(combined_dataset)}")

        # 4. 对合并后的数据集进行 shuffle（随机打乱）
        # 使用一个固定的 seed (例如 42) 可以保证每次运行结果一致
        print("正在打乱数据集...")
        shuffled_dataset = combined_dataset.shuffle(seed=42)
        # shuffled_dataset = combined_dataset.shuffle(seed=56)

        # 5. 选取前 num_samples 个样本
        num_to_select = min(num_samples, len(shuffled_dataset))
        random_sample = shuffled_dataset.select(range(num_to_select))

        print(f"\n成功抽取 {len(random_sample)} 个随机问题。")
        return random_sample

    except Exception as e:
        print(f"发生错误: {e}")
        return None

# --- 执行脚本 ---
random_samples = get_random_mmlu_samples(10)

if random_samples:
    print("\n--- 抽样问题示例 (前5个) ---")
    
    # 遍历并打印前5个抽取的样本
    for i, example in enumerate(random_samples):
        if i >= 5: # 只显示5个作为演示
            break
            
        print(f"\n[问题 {i+1}] (科目: {example['subject']})")
        print(f"Q: {example['question']}")
        
        choices = example['choices']
        print(f"  A: {choices[0]}")
        print(f"  B: {choices[1]}")
        print(f"  C: {choices[2]}")
        print(f"  D: {choices[3]}")
        
        # 答案 'answer' 是一个索引 (0, 1, 2, 或 3)
        correct_choice = ['A', 'B', 'C', 'D'][example['answer']]
        print(f"A: {correct_choice} (索引: {example['answer']})")

    # 你可以将这100个样本保存到文件
    # random_samples.to_json("mmlu_100_random_samples.jsonl")
    # print("\n已将100个样本保存到 'mmlu_100_random_samples.jsonl'")
    
file_path_json = "mmlu_10_random_samples.jsonl"

try:
    # 调用 to_json() 方法
    random_samples.to_json(file_path_json)
    
    print(f"\n成功将100个随机样本保存到 JSON Lines 文件: {file_path_json}")

except Exception as e:
    print(f"保存到 JSON 失败: {e}")

正在获取 MMLU 科目列表...
共找到 58 个科目。
正在加载所有科目的 'test' 数据集...
加载科目 auxiliary_train 失败: Unknown split "test". Should be one of ['train'].
正在合并所有数据集...
数据集合并完毕，总问题数: 14042
正在打乱数据集...

成功抽取 10 个随机问题。

--- 抽样问题示例 (前5个) ---

[问题 1] (科目: college_physics)
Q: Positronium is an atom formed by an electron and a positron (antielectron). It is similar to the hydrogen atom, with the positron replacing the proton. If a positronium atom makes a transition from the state with n=3 to a state with n=1, the energy of the photon emitted in this transition is closest to
  A: 6.0 e
  B: 6.8 eV
  C: 12.2 eV
  D: 13.6 eV
A: A (索引: 0)

[问题 2] (科目: anatomy)
Q: The regional lymphatic drainage of the left side of the tip of the tongue is to the
  A: left submental lymph node.
  B: left and right submental lymph nodes.
  C: left submandibular lymph node.
  D: left and right submandibular lymph nodes.
A: B (索引: 1)

[问题 3] (科目: high_school_statistics)
Q: A school board of a large school district is proposing a new dress cod

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


成功将100个随机样本保存到 JSON Lines 文件: mmlu_10_random_samples.jsonl
